# Index Table Preparation

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
import json
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit


# config
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

Pandas notes for me: 

gdf.loc[ROWS, COLUMNS] = "select rows by this value, and columns by this value"

new field:
gdf["new_field"] = (
    gdf["field_a"].astype(str)
    + "_"
    + gdf["field_b"].astype(str)
    + "_"
    + gdf["field_c"].astype(str)
)

### Load the Shapefile and give it a true unique ID before doing anything else to it. 

In [ ]:
# this is the path to the original feature service, downloaded as a shapefile from here: 
# https://www.arcgis.com/home/item.html?id=f260e4065a384e7ebee05b23362f380f#overview
# Layer 36112 - https://gis5.uspatial.umn.edu/arcgis/rest/services/mhapo/MHAPO_dynamic/MapServer/0
original_index_path = "data/historic_airphoto_index/shapefiles/umn_airphoto_index_original.shp"

gdf_original = gpd.read_file(original_index_path)

print(gdf_original.crs)
# we're keeping this in web mercator for now. 

In [ ]:
# make a unique identifier

gdf_original["source_id"] = range(1, len(gdf_original) + 1)

assert gdf_original["source_id"].is_unique
assert gdf_original["source_id"].notna().all()

In [ ]:
gdf_original

In [ ]:
gdf_original.to_file(
    "data/historic_airphoto_index/umn_index_original_with_source_id.gpkg",
    driver="GPKG"
)

### Cleaning the original table

In [ ]:
gdf_clean = gdf_original.copy()

#preview the columns
gdf_clean.columns

#### Cleaning the URLs
And looking for true duplicates...

In [ ]:
# clean URL spaces

gdf_clean["url_link"] = gdf_clean["url_link"].str.strip()

In [ ]:
exists = 'https://geo.lib.umn.edu/Olmsted/1940/WB-3A-115.jpg' in gdf_clean['url_link'].values

exists

In [ ]:
# identify duplicate URLs

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [ ]:
# identify duplicate URLs with unique photo_ids to be fixed

fake_duplicates = gdf_clean[
    gdf_clean["url_link"].duplicated(keep=False)
    & ~gdf_clean["photo_id"].duplicated(keep=False)]
fake_duplicates = fake_duplicates.sort_values(by=['flight_cod','url_link'])
fake_duplicates

In [ ]:
# fix olmsted county URLs

bad_url = "https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg"

mask = gdf_clean["url_link"] == bad_url

gdf_clean.loc[mask, "url_link"] = (
    "https://geo.lib.umn.edu/Olmsted/1940/"
    + gdf_clean.loc[mask, "photo_id"]
    + ".jpg"
)

In [ ]:
check = gdf_clean[gdf_clean["flight_cod"] == "Olmsted"]

check

In [ ]:
# identify duplicate URLs again

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [ ]:
# drop duplicates arbitrarily (the only thing that varies is the geometry and we're not using it.)
gdf_clean = gdf_clean.drop_duplicates(
    subset=["url_link"],
    keep="first"
).copy()

In [ ]:
# identify duplicate URLs again
# there should be none. 

duplicates = gdf_clean[gdf_clean["photo_id"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','photo_id'])
duplicates

In [ ]:
print("Unique URLs:", gdf_clean["url_link"].is_unique)
print("Remaining records:", len(gdf_clean))

### Clean up other values

In [ ]:
# convert year field to an integer rather than a float
# handles missing values
gdf_clean["year_"] = gdf_clean["year_"].astype("Int64")

In [ ]:
# populate missing counties from the flight code field
gdf_clean["name10"] = gdf_clean["name10"].fillna(
    gdf_clean["flight_cod"]
)

In [ ]:
# check for nulls
print(gdf_clean["name10"].isna().any())

print(gdf_clean["year_"].isna().any())

print(gdf_clean["photo_id"].isna().any())

print(gdf_clean["url_link"].isna().any())

### Adding filename identifier and validation

In [ ]:
# add a new unique, human readable identifier field called "cty_year_id"
# it will be used for naming files
# it is a combination of the county, the year, and the photo ID. 

gdf_clean["cty_year_id"] = (
    gdf_clean["name10"].str.strip().str.lower().str.replace(" ", "_").str.replace('.', '', regex=False)
    + "_"
    + gdf_clean["year_"].astype("string")
    + "_"
    + gdf_clean["photo_id"].str.strip().str.lower().str.replace(" ", "_")
)

# add a number to any non-unique values produced by this
counts = gdf_clean.groupby("cty_year_id").cumcount()

gdf_clean["cty_year_id"] += counts.map(
    lambda n: f"_{n}" if n > 0 else ""
)

In [ ]:
# validate that the periods were handled correctly
gdf_clean_filtered = gdf_clean.loc[gdf_clean["name10"] == "St. Louis"]

gdf_clean_filtered

In [ ]:
# validate that this new field is unique and contains no nulls
# should return 

print(f"All unique values:", gdf_clean['cty_year_id'].is_unique)
print(f"Has null values:", gdf_clean['cty_year_id'].isna().any())
print(f"Has spaces:", gdf_clean['cty_year_id'].str.contains(' ').any())

In [ ]:
# check it out
gdf_clean

In [ ]:
# save it
gdf_clean.to_file("data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg")

### Drop and reorder columns

In [ ]:
# open cleaned table
path = "data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg"

gdf = gpd.read_file(path)

In [ ]:
gdf

In [ ]:
# inspect
gdf.columns

In [ ]:
# rename
new_gdf = gdf.rename(columns={
    "url_link" : "url",
    "year_" : "year",
    "name10" : "county",
    'flight_cod' : 'flight_code'
})

In [ ]:
# add
new_gdf["worldfile_json"] = None
new_gdf["h_extent"] = None
new_gdf["scale"] = None
new_gdf["dpi_x"] = None
new_gdf["width_px"] = None
new_gdf["status"] = None
new_gdf["gcp_count"] = None
new_gdf["confidence"] = None

In [ ]:
new_gdf = new_gdf[['cty_year_id','status','county','year','photo_id','h_extent','gcp_count','confidence','scale','dpi_x','width_px','worldfile_json','url','flight_code','source_id','geometry']]

new_gdf

In [ ]:
new_gdf = new_gdf.sort_values(by=['county','year'], ascending=[True, True])

new_gdf

In [ ]:
new_gdf.to_file("data/historic_airphoto_index/index_table_cleaned_pre-json.gpkg")

### Get JPEG world file affine parameters, if they exist

In [ ]:
gdf = gpd.read_file("data/historic_airphoto_index/index_table_cleaned_pre-json.gpkg")

In [ ]:
def fetch_worldfile(jpg_url):
    """Retrieve a JGW and return its affine parameters as JSON."""

    if not isinstance(jpg_url, str) or not jpg_url:
        return None

    # Replace .jpg with .jgw
    jgw_url = jpg_url.removesuffix(".jpg") + ".jgw"

    try:
        response = requests.get(jgw_url, timeout=15)

        # Confirmed missing world file
        if response.status_code in (300, 404):
            return "no jgw"

        response.raise_for_status()

        # Some missing files may return an HTML page
        if "text/html" in response.headers.get("Content-Type", "").lower():
            return "no jgw" if "Multiple Choices" in response.text else None

        # World-file parameter order: A, D, B, E, C, F
        values = [float(v) for v in response.text.split()]

        if len(values) != 6:
            raise ValueError("World file must contain exactly 6 values")

        parameters = dict(zip(
            ["A", "D", "B", "E", "C", "F"],
            values
        ))

        return json.dumps(parameters)

    except (requests.RequestException, ValueError) as e:
        print(f"Failed to retrieve {jgw_url}: {e}")
        return None

In [ ]:
test_json = fetch_worldfile("https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1989/stl_028_107.jpg")

print(test_json)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd

output_path = Path(
    "data/historic_airphoto_index/index_with_jgw_jsons.gpkg"
)

# Resume from previous saved progress, if available
if output_path.exists():
    gdf = gpd.read_file(output_path)
    print("Loaded previously saved progress.")

# Find records that haven't been checked
pending = gdf.loc[gdf["worldfile_json"].isna(), "url"]

print(f"Remaining URLs to check: {len(pending):,}")

save_every = 500
completed = 0

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(fetch_worldfile, url): idx
        for idx, url in pending.items()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Fetching world files"
    ):
        idx = futures[future]

        # Store the result immediately
        gdf.at[idx, "worldfile_json"] = future.result()

        completed += 1

        # Save progress every 500 requests
        if completed % save_every == 0:
            gdf.to_file(output_path, driver="GPKG")

# Final save
gdf.to_file(output_path, driver="GPKG")

print(gdf["worldfile_json"].notna().value_counts())

### Export

In [ ]:
gdf.to_file("data/historic_airphoto_index/main_airphoto_index.gpkg")